# App Service, Functions & Containers

If the previous notebook was about renting hardware, this one is about renting *runtimes*. Azure's PaaS and container portfolio takes everything above the OS — the web server, the language runtime, the orchestrator — and offers it as a managed surface. You bring code, sometimes a container; Microsoft handles the rest.

The portfolio sits on a continuum. On one end, **App Service** runs your web app from a git push and hands you a TLS-fronted domain in minutes. On the other end, **AKS** gives you a full Kubernetes cluster with raw flexibility and the operational weight to match. In between are **Functions**, **Container Apps**, **Container Instances**, and the **Container Registry** that feeds them all. The art is picking the right point on the continuum — too high a layer and you fight the platform; too low and you've signed up for ops work you didn't want.

## App Service plans

**Azure App Service** runs web apps, REST APIs, and mobile backends in .NET, Java, Node.js, Python, PHP, and Ruby. Every App Service runs inside an **App Service plan** — the billing and capacity object. The plan is the VM (or set of VMs); the apps are the workloads riding on it. One plan can host many apps, which share its CPU and memory.

Tiers, in order of capability:

- **Free / Shared** — multi-tenant, no SLA, time-quota'd. Demos only.
- **Basic** — dedicated VMs, manual scale-out, no slots. Dev/test.
- **Standard** — slots (5), autoscale, daily backups. The classic production tier.
- **Premium v3 (P0v3–P3v3)** — faster VMs, more slots (20), VNet integration, zone redundancy. Most new production workloads.
- **Isolated v2 (App Service Environment v3)** — single-tenant, deployed into your VNet. For workloads that cannot share a multi-tenant front-end (regulated, ultra-large, traffic-isolated).

Two scaling dimensions:

- **Scale up** — change the plan tier or size. Bigger VMs, more memory, more features. Restart-impacting.
- **Scale out** — add more identical instances of the plan. Zero-impact when behind the built-in load balancer. Drives the autoscale rules.

AWS equivalent: App Service Standard ≈ Elastic Beanstalk; Premium v3 with VNet integration ≈ Beanstalk + VPC config; Isolated v2 ≈ Beanstalk in a dedicated VPC. The closer analogue to App Service's developer experience is actually App Runner.

## Slots, domains, TLS, VNet integration

Four App Service features that you will use on essentially every production app.

**Deployment slots** are independent App Service instances that share the plan. You deploy a new version to a `staging` slot, run smoke tests against `myapp-staging.azurewebsites.net`, and then **swap** with production. Swap is near-instantaneous — Azure flips internal DNS — and lets you roll back by swapping again. Slot-specific settings (connection strings, app settings marked `slot setting`) stay pinned, so the swap doesn't accidentally point staging credentials at prod.

**Custom domains and TLS.** Apps get a default `*.azurewebsites.net` hostname; you add your own via DNS CNAME or TXT validation. App Service issues and renews **free managed TLS certificates** for verified domains — no Key Vault, no cron job. Bring-your-own certs are supported when you need an EV cert or a wildcard from a specific CA.

**VNet integration** lets the app reach private resources — Azure SQL via private endpoint, a Redis Cache, an internal API behind a Private Link — without going through the public internet. Two flavours: **regional VNet integration** (the default, app reaches into a delegated subnet) and **gateway-required** (older, for App Service Environments). Combined with **private endpoints on the app itself**, you can take the app fully off the public internet — inbound and outbound — for sensitive workloads.

**Hybrid Connections** let the app dial back to an on-prem service over a relay; useful for migrations where the database is still on-site.

## Azure Functions — event-driven code

**Azure Functions** is FaaS: you write a function, you bind it to a *trigger* and one or more *bindings*, and Azure runs it on demand. The trigger fires the function (HTTP request, queue message, blob upload, timer); bindings inject inputs and accept outputs (read a Cosmos DB document, write a Service Bus message) without you writing connection code.

Hosting plans determine performance and price:

- **Consumption** — pure pay-per-execution. Scales from zero to many instances automatically. Subject to **cold starts** (sub-second to a few seconds depending on language and dependencies). The cheap default for sporadic workloads.
- **Flex Consumption** — newer consumption tier with pre-warmed instances and VNet integration baked in. Eliminates the cold-start floor without paying for always-on. Use this for new consumption workloads where it's available.
- **Premium (EP-series)** — pre-warmed instances, VNet integration, unlimited execution duration, larger memory. For latency-sensitive APIs that can't tolerate cold starts.
- **Dedicated (App Service plan)** — run Functions on an existing App Service plan you already pay for. Best when you have spare capacity.

**Durable Functions** is an extension for orchestrating long-running, stateful workflows in code. You write an *orchestrator function* that calls *activity functions* and awaits their results; the framework persists state to storage at each `await`, so the orchestrator can run for hours or days, survive restarts, and replay deterministically. Patterns it makes easy: function chaining, fan-out/fan-in, async HTTP polling, human-interaction (wait-for-external-event).

AWS comparison: Consumption ≈ Lambda; Premium ≈ Lambda with provisioned concurrency; Durable Functions ≈ Step Functions written as code.

In [ ]:
# Create an App Service and a Function App side by side.

RG=rg-paas-demo
LOC=eastus
az group create --name $RG --location $LOC

# 1. App Service — Linux, Premium v3, with one app and a staging slot.
az appservice plan create \
  --resource-group $RG --name plan-web \
  --sku P1v3 --is-linux

az webapp create \
  --resource-group $RG --plan plan-web \
  --name myapp-foundations --runtime "PYTHON:3.11"

az webapp deployment slot create \
  --resource-group $RG --name myapp-foundations --slot staging

# Swap staging into production after smoke tests pass.
az webapp deployment slot swap \
  --resource-group $RG --name myapp-foundations --slot staging --target-slot production

# 2. Function App — Flex Consumption, Python.
az storage account create \
  --resource-group $RG --name stfuncdemo$RANDOM \
  --location $LOC --sku Standard_LRS

az functionapp create \
  --resource-group $RG --name func-foundations-demo \
  --storage-account stfuncdemo$RANDOM \
  --flexconsumption-location $LOC \
  --runtime python --runtime-version 3.11

az group delete --name $RG --yes --no-wait

## Container Instances — the quickest container

**Azure Container Instances (ACI)** runs a single container (or a small group) on demand. No cluster, no orchestrator, no node management — you give it an image and a CPU/memory size and it runs. Per-second billing.

What it's for:

- **Burst capacity** for an AKS cluster via the **virtual node** integration (AKS schedules pods onto ACI when the cluster is full).
- **Short-lived batch jobs** that don't need a scheduler.
- **Build agents** spawned by a CI/CD pipeline that vanish after the job.
- **Quick CLI demos** where Kubernetes is overkill.

What it isn't:

- A long-running web app — there's no autoscale, no load balancer, no zero-downtime deploy.
- A microservices platform — no service discovery, no orchestration.

AWS equivalent: ACI ≈ AWS Fargate run via `RunTask`, without the ECS cluster wrapper. If you want a fuller managed-container platform, look at Container Apps next.

## Azure Kubernetes Service

**AKS** is managed Kubernetes. Microsoft runs the control plane (API server, scheduler, etcd) for free and bills you for the worker VMs. You get an upstream-compatible Kubernetes cluster with Azure-aware integrations: AAD/Entra ID for auth, Azure CNI for networking, Azure Disk and File for persistent volumes, Azure Monitor for observability.

**Node pools** group nodes by VM SKU and purpose:

- **System node pool** — runs cluster-critical pods (CoreDNS, metrics-server). At least one is mandatory. Use a stable, modest SKU.
- **User node pools** — run your workloads. You can have many, each on a different VM family. GPU nodes for ML, memory-heavy nodes for caches, spot nodes for cheap batch.

Two **networking modes**, and the choice is permanent:

- **kubenet** — pods get private IPs from a separate CIDR, NAT'd through the node's IP. Lighter address-space use, but limits some features.
- **Azure CNI** — pods get IPs directly from the VNet subnet. First-class VNet citizens; needed for VNet-aware security tools. Eats subnet space fast — plan a /22 or wider.
- **Azure CNI Overlay** — newer hybrid: nodes are on the VNet, pods on an overlay network. Solves the address-space problem without losing CNI features. The new default for large clusters.

Two autoscalers run together:

- **Cluster Autoscaler** adds/removes *nodes* when pods can't be scheduled (or nodes sit idle).
- **Horizontal Pod Autoscaler (HPA)** adds/removes *pods* based on CPU/memory or custom metrics.
- **KEDA** (Kubernetes Event-driven Autoscaler) extends HPA to scale on external signals — Service Bus queue depth, Event Hubs lag, Kafka offset, even cron schedules. KEDA is built into AKS and is what enables scale-to-zero patterns.

**Ingress** comes from add-ons: the **AKS-managed application routing** add-on (NGINX under the hood), or **Application Gateway Ingress Controller (AGIC)** if you want the L7 WAF in front. Both work; AGIC offloads the data plane to a managed Azure resource.

AKS pulls in real ops weight: Kubernetes upgrades every few months, certificate rotation, node OS patching (semi-managed via node-image upgrades), RBAC binding between Entra ID and Kubernetes RBAC. Reach for it when you need its flexibility; default to Container Apps if you don't.

## Container Apps — Kubernetes without the cluster

**Azure Container Apps** is the serverless container platform: scale-to-zero, KEDA-driven autoscale, traffic-splitting between revisions, and built-in Dapr — all without you ever seeing the underlying Kubernetes.

The model:

- A **Container App Environment** is the trust and networking boundary (one VNet, one Log Analytics workspace). Multiple Container Apps share it.
- A **Container App** is a service: a container image, scale rules, ingress config, and **revisions** (immutable snapshots of the config). You can split traffic 90/10 across revisions for blue/green or canary.
- **Scale rules** speak KEDA: scale on HTTP concurrency, queue depth, custom metric, or cron. Scale to zero when idle (you pay nothing until the next request).
- **Dapr** sidecars are first-class — set `dapr.enabled = true` and you get service invocation, pub/sub, state store, and bindings with zero infra setup.

The trade-off vs AKS: less control. You don't get to write raw Kubernetes manifests, you can't run privileged sidecars, you can't tune the kubelet. The trade-off vs App Service: containers, not buildpacks. You give up the language-runtime conveniences but get to ship any image.

Container Apps is the right default for new microservices in 2026. Drop down to AKS only when a specific Kubernetes feature forces you there.

AWS comparison: Container Apps ≈ ECS Fargate + CodeDeploy traffic shifting, with Dapr as the value-add.

## Azure Container Registry

**Azure Container Registry (ACR)** is the OCI registry. Three tiers:

- **Basic** — single region, dev/test.
- **Standard** — production-grade with higher throughput and storage.
- **Premium** — adds **geo-replication** (multi-region replicas with one logical name), **content trust** (signed images), **private endpoints**, customer-managed encryption keys, and **tasks** (build, test, patch on schedule or on git push).

Two features earn their keep in real deployments:

- **Geo-replication** — push once to `myregistry.azurecr.io`, get pull-local-fast in every replicated region. AKS clusters in different regions pull from the nearest replica automatically. Premium-only.
- **ACR Tasks** — run `docker build` in the cloud on a git push or on a schedule. The killer pattern is **base-image update tasks**: when Microsoft publishes a new patched runtime image, the task rebuilds every downstream image that uses it as a base, then re-pushes. Your fleet stays patched without anyone touching a Dockerfile.

Authentication for AKS/Container Apps to pull from ACR is best done with **managed identity** + the `AcrPull` role — no admin user, no credentials in pipelines.

## Choosing the right surface

The portfolio looks crowded until you map it to a decision tree:

```
Is the workload event-driven, short-lived,
and happy with cold starts?
  └── yes → Functions (Consumption or Flex)

Is it a web app or REST API in a supported runtime,
with no special container needs?
  └── yes → App Service

Is it a containerised microservice that should
scale to zero and use Dapr?
  └── yes → Container Apps

Do you need raw Kubernetes — operators, custom CRDs,
DaemonSets, privileged sidecars, hand-tuned scheduling?
  └── yes → AKS

One-off container, batch job, AKS burst node?
  └── yes → Container Instances
```

The mistake everyone makes once: reaching for AKS for a three-service application. The cluster is real ops work, and the savings on managed runtimes (App Service, Container Apps, Functions) come from *not paying* that operational tax. Treat AKS as the high-ceiling default for platforms big enough to amortise the cluster, and let the smaller patterns sit on the lighter surfaces above it.